# Recommender Systems - Part A Project
**Paper:** SVD-GoRank: Recommender System Algorithm Using SVD and Gower's Ranking

**Dataset:** Flixster

**Team Members:** Harun Korkmaz, Muhammet Salih Hasılcıo, Orhan Efe Bayrak, Muhiddin Fırat, Yasin Furkan Abasız, Yakup Berkay Genceroğlu, Görkem Yahya Bakan

This notebook contains data preprocessing and matrix construction steps for the Flixster social movie recommendation dataset obtained from Figshare. With over 8 million ratings, it is critical for testing SVD scalability.

# ==========================================
# PROGRESS 1: DATA LOADING & PREPROCESSING
# ==========================================

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

file_path_flix = r"C:\Users\Harun\Desktop\üni\4.SINIF\2.donem\recommender\dataset\Flixster-dataset\data\edges.csv"

# Read the edges.csv file
df_flix = pd.read_csv(file_path_flix)

# ERROR FIX: Only 2 columns, so we take first 2
df_flix = df_flix.iloc[:, :2]
df_flix.columns = ['user_id', 'item_id']

# Rating column is required for SVD! Since this is implicit feedback,
# assign value '1' to all interactions
df_flix['rating'] = 1

# Convert IDs to integers for safety
df_flix['user_id'] = df_flix['user_id'].astype(int)
df_flix['item_id'] = df_flix['item_id'].astype(int)

# Calculate statistics
n_users_flix = df_flix['user_id'].nunique()
n_items_flix = df_flix['item_id'].nunique()
n_ratings_flix = len(df_flix)
sparsity_flix = 1.0 - (n_ratings_flix / (n_users_flix * n_items_flix))

print("--- FLIXSTER - PROGRESS 1 ---")
print(f"Total Users: {n_users_flix}")
print(f"Total Movies: {n_items_flix}")
print(f"Total Interactions (Ratings): {n_ratings_flix}")
print(f"Sparsity Ratio: {sparsity_flix:.6f} ({sparsity_flix*100:.4f}%)")
print("\nFirst 5 Rows of Simplified Dataset (Rating column set to 1 by us):")
display(df_flix.head())

--- FLIXSTER - PROGRESS 1 ---
Total Users: 99804
Total Movies: 2523386
Total Interactions (Ratings): 9197336
Sparsity Ratio: 0.999963 (99.9963%)

First 5 Rows of Simplified Dataset (Rating column set to 1 by us):


,user_id,item_id,rating
0,100000,14106,1
1,100000,14133,1
2,100000,157817,1
3,100000,15856,1
4,100000,1610,1


# ==========================================
# PROGRESS 2: TRAIN/TEST SPLIT & MATRIX
# ==========================================

In [ ]:
# Split data into 80% Training and 20% Testing
train_flix, test_flix = train_test_split(df_flix, test_size=0.20, random_state=42)

print("\n--- FLIXSTER - PROGRESS 2 ---")
print(f"Training Set Size: {len(train_flix)} rows")
print(f"Test Set Size: {len(test_flix)} rows\n")

print("CRITICAL ENGINEERING NOTE")
print("Flixster matrix is enormous in size. Pandas would exceed memory limits,")
print("so we create a representative 'Sample Matrix' with only top 500 users and 500 movies.\n")

# To show the professor, create downsampled example data
top_users_flix = train_flix['user_id'].value_counts().index[:500]
top_items_flix = train_flix['item_id'].value_counts().index[:500]

sample_df_flix = train_flix[train_flix['user_id'].isin(top_users_flix) & train_flix['item_id'].isin(top_items_flix)]

# Use pivot_table for possible duplicate records (same user clicked movie twice)
sample_matrix_flix = sample_df_flix.pivot_table(
    index='user_id', 
    columns='item_id', 
    values='rating',
    aggfunc='mean'
)

print(f"Sample Matrix Size: {sample_matrix_flix.shape}")
print("Sample Matrix Visualization (Empty cells are NaN):")
display(sample_matrix_flix.head())


--- FLIXSTER - PROGRESS 2 ---
Training Set Size: 7357868 rows
Test Set Size: 1839468 rows

CRITICAL ENGINEERING NOTE
Flixster matrix is enormous in size. Pandas would exceed memory limits,
so we create a representative 'Sample Matrix' with only top 500 users and 500 movies.

Sample Matrix Size: (410, 363)
Sample Matrix Visualization (Empty cells are NaN):


item_id,17,21,26,31,48,57,94,139,164,184,...,81095,86444,87065,87441,90076,118106,135364,147041,150519,158121
user_id,,,,,,,,,,,,,,,,,,,,,
831,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1383,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1939,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2069,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
